# Model Validation Metrics (Loss + Accuracy)

This notebook evaluates trained checkpoints (`best.pt` / `last.pt`) on a validation split and reports:
- Validation loss
- Perplexity
- Token-level top-1 accuracy
- Token-level top-5 accuracy

In [ ]:
from __future__ import annotations

import json
import math
from pathlib import Path

import torch
import torch.nn.functional as F

from train_gpt import TrainConfig, MusicGPT, split_data

In [ ]:
# --- Config ---
PROJECT_ROOT = Path("..").resolve()  # notebook is in src/
DATASET_PATH = PROJECT_ROOT / "tokenized" / "dataset.jsonl"
BEST_CKPT = PROJECT_ROOT / "src" / "best.pt"
LAST_CKPT = PROJECT_ROOT / "src" / "last.pt"

SEED = 42
VAL_RATIO = 0.03
EVAL_BATCHES = 30

print("Dataset exists:", DATASET_PATH.exists())
print("best.pt exists:", BEST_CKPT.exists())
print("last.pt exists:", LAST_CKPT.exists())

In [ ]:
def load_dataset_ids(path: Path) -> list[list[int]]:
    sequences: list[list[int]] = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            row = json.loads(line)
            ids = row.get("token_ids")
            if isinstance(ids, list) and len(ids) >= 2:
                sequences.append([int(x) for x in ids])
    if not sequences:
        raise RuntimeError(f"No usable sequences in {path}")
    return sequences


def sample_batch_ids(
    sequences: list[list[int]],
    batch_size: int,
    seq_len: int,
    device: torch.device,
) -> tuple[torch.Tensor, torch.Tensor]:
    valid = [s for s in sequences if len(s) > seq_len]
    if not valid:
        raise RuntimeError(f"No sequences long enough for seq_len={seq_len}")

    x_list, y_list = [], []
    for _ in range(batch_size):
        seq = valid[torch.randint(0, len(valid), (1,)).item()]
        start = torch.randint(0, len(seq) - seq_len, (1,)).item()
        chunk = seq[start : start + seq_len + 1]
        x_list.append(torch.tensor(chunk[:-1], dtype=torch.long))
        y_list.append(torch.tensor(chunk[1:], dtype=torch.long))

    x = torch.stack(x_list, dim=0).to(device)
    y = torch.stack(y_list, dim=0).to(device)
    return x, y

In [ ]:
@torch.no_grad()
def evaluate_checkpoint(
    ckpt_path: Path,
    val_sequences: list[list[int]],
    eval_batches: int,
    device: torch.device,
) -> dict:
    ckpt = torch.load(ckpt_path, map_location="cpu")
    cfg = TrainConfig(**ckpt["config"])

    model = MusicGPT(cfg)
    model.load_state_dict(ckpt["model_state"])
    model.to(device)
    model.eval()

    use_amp = device.type == "cuda"

    total_loss = 0.0
    total_tokens = 0
    top1_correct = 0
    top5_correct = 0

    for _ in range(eval_batches):
        x, y = sample_batch_ids(val_sequences, cfg.batch_size, cfg.max_seq_len, device)

        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), reduction="mean")

        total_loss += float(loss.item())

        # Accuracy metrics
        pred_top1 = torch.argmax(logits, dim=-1)
        top1_correct += int((pred_top1 == y).sum().item())

        top5 = torch.topk(logits, k=min(5, logits.size(-1)), dim=-1).indices
        y_expanded = y.unsqueeze(-1)
        top5_correct += int((top5 == y_expanded).any(dim=-1).sum().item())

        total_tokens += y.numel()

    mean_loss = total_loss / eval_batches
    ppl = math.exp(mean_loss)
    top1_acc = top1_correct / max(1, total_tokens)
    top5_acc = top5_correct / max(1, total_tokens)

    return {
        "checkpoint": ckpt_path.name,
        "step": ckpt.get("step", -1),
        "val_loss": mean_loss,
        "perplexity": ppl,
        "top1_accuracy": top1_acc,
        "top5_accuracy": top5_acc,
        "eval_batches": eval_batches,
        "tokens_evaluated": total_tokens,
    }

In [ ]:
# Build split matching training
sequences = load_dataset_ids(DATASET_PATH)
_, val_sequences = split_data(sequences, val_ratio=VAL_RATIO, seed=SEED)

print(f"Total sequences: {len(sequences):,}")
print(f"Validation sequences: {len(val_sequences):,}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
results = []
for ckpt in [BEST_CKPT, LAST_CKPT]:
    if ckpt.exists():
        results.append(
            evaluate_checkpoint(
                ckpt_path=ckpt,
                val_sequences=val_sequences,
                eval_batches=EVAL_BATCHES,
                device=device,
            )
        )
    else:
        print(f"Missing checkpoint: {ckpt}")

if not results:
    raise RuntimeError("No checkpoints found to evaluate")

print("\nValidation Metrics")
print("=" * 80)
for r in results:
    print(f"Checkpoint:   {r['checkpoint']}")
    print(f"Step:         {r['step']}")
    print(f"Val loss:     {r['val_loss']:.4f}")
    print(f"Perplexity:   {r['perplexity']:.2f}")
    print(f"Top-1 acc:    {r['top1_accuracy']*100:.2f}%")
    print(f"Top-5 acc:    {r['top5_accuracy']*100:.2f}%")
    print(f"Eval batches: {r['eval_batches']} | Tokens: {r['tokens_evaluated']:,}")
    print("-" * 80)

best_by_loss = min(results, key=lambda x: x['val_loss'])
print(f"Best checkpoint by val loss: {best_by_loss['checkpoint']}")

## How to Interpret

- Lower **val loss** and **perplexity** are better.
- Higher **top-1/top-5 accuracy** are better.
- Use these metrics to choose between `best.pt` and `last.pt` for generation.

Rule of thumb:
- If `last.pt` is worse than `best.pt`, generate from `best.pt`.
- If they are very close, either is fine; keep using `best.pt` as default.